# African Wildlife Object Detector (YOLOv8n)

This notebook runs end-to-end from a fresh Google Colab runtime:
1. Installs pinned dependencies
2. Downloads, deduplicates, and inspects the African Wildlife dataset
3. Trains a baseline YOLOv8n detector (`imgsz=416`)
4. Runs a controlled experiment (`imgsz=640`)
5. Compares validation results, selects the best checkpoint, evaluates on the held-out test set, and visualizes predictions and failure cases

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/NurkhanZ/DeepLearning2026.git"

if not Path("scripts/train.py").exists():
    if not Path("DeepLearning2026").exists():
        !git clone {REPO_URL}
    os.chdir("DeepLearning2026")

!pip install -q -r requirements.txt
print("Working directory:", os.getcwd())

In [ ]:
import torch
import ultralytics

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Prepare and Inspect Dataset

In [ ]:
!python scripts/prepare_data.py

In [ ]:
from IPython.display import Image, display

display(Image(filename="results/dataset_samples/ground_truth_grid.png"))

## 2. Train Baseline Detector

In [ ]:
!python scripts/train.py --config configs/baseline.yaml

In [ ]:
display(Image(filename="results/baseline/results.png"))

## 3. Controlled Experiment

**Hypothesis:** Increasing input image resolution from `416` to `640` while keeping all other settings fixed will preserve finer spatial details for distant or partially occluded animals, improving validation recall and `mAP@0.50` at the cost of longer training time.

In [ ]:
!python scripts/train.py --config configs/experiment.yaml

In [ ]:
display(Image(filename="results/experiment/results.png"))

## 4. Compare Validation Results and Select Best Checkpoint

In [ ]:
import json
import shutil
import pandas as pd

baseline = json.loads(Path("results/baseline/metrics.json").read_text())
experiment = json.loads(Path("results/experiment/metrics.json").read_text())

comparison = pd.DataFrame([
    {
        "Run": r["run_name"],
        "imgsz": r["imgsz"],
        "Val Precision": r["val_precision"],
        "Val Recall": r["val_recall"],
        "Val mAP@0.50": r["val_map50"],
        "Train Time (s)": r["training_time_sec"],
    }
    for r in [baseline, experiment]
])
display(comparison)

best_run = experiment if experiment["val_map50"] >= baseline["val_map50"] else baseline
Path("weights").mkdir(exist_ok=True)
shutil.copy2(best_run["best_weights"], "weights/best.pt")
best_imgsz = best_run["imgsz"]
print(f"Selected run: {best_run['run_name']} (imgsz={best_imgsz}, val mAP@0.50={best_run['val_map50']})")

## 5. Final Evaluation on Held-Out Test Set

In [ ]:
!python scripts/evaluate.py --weights weights/best.pt --imgsz {best_imgsz} --conf 0.25 --iou 0.50

In [ ]:
display(Image(filename="results/test_evaluation/test_predictions_grid.png"))
for i in range(1, 4):
    path = Path(f"results/test_evaluation/failure_{i}.png")
    if path.exists():
        display(Image(filename=str(path)))